# 05 — Surge Mechanism Validation

Validates the synthetic surge injection used for the operator-hint experiments
(notebooks 06–07). A surge multiplies the arrival rate for a short window; if
the agent cannot scale up fast enough, the job queue floods and SLA breaches
follow. This notebook confirms surges create genuine, recoverable-or-not stress,
and saves the surge time-series for the report figure.

Uses `env.py` (surge support built in).

## 1. Run a week with a surge and record the response

A fixed policy (hold VMs steady) is used so the surge's effect on the queue and
breaches is visible without the agent masking it.

In [1]:
from env import CloudClusterEnv, STEPS_PER_WEEK
import json, numpy as np

stats = json.load(open('trace_params.json'))['stats']

# enable surges; fixed seed for a reproducible, documented surge
env = CloudClusterEnv(stats, enable_surges=True, enable_hints=False, seed=42)
env.reset()
print("Scheduled surge(s) this episode:", env.surges, "\n")

# run the week holding VMs steady (action 0), record queue + breaches per step
queue_series, breach_series, surge_series = [], [], []
for t in range(STEPS_PER_WEEK):
    obs, r, done, tr, info = env.step(np.array([0.0]))
    queue_series.append(info['queue'])
    breach_series.append(info['breaches'])
    surge_series.append(info['surge_mult'])
    if info['surge_mult'] > 1.0:
        print(f"step {t}: SURGE x{info['surge_mult']:.1f} | "
              f"queue {info['queue']} | breaches {info['breaches']}")
    if done: break

# save the time-series for the report figure
json.dump({'queue': queue_series, 'breaches': breach_series, 'surge': surge_series,
           'surges': env.surges},
          open('surge_timeseries.json', 'w'))
print("\nSaved surge_timeseries.json")
print(f"Peak queue during surge: {max(queue_series)}")
print(f"Total breaches from surge: {sum(breach_series)}")

Setup complete. Steps per week: 672
CloudClusterEnv defined.
Scheduled surge(s) this episode: [{'start': 509, 'end': 520, 'magnitude': 4.787896879867073}] 

step 508: SURGE x4.8 | queue 0 | breaches 0
step 509: SURGE x4.8 | queue 111 | breaches 0
step 510: SURGE x4.8 | queue 268 | breaches 0
step 511: SURGE x4.8 | queue 398 | breaches 0
step 512: SURGE x4.8 | queue 597 | breaches 0
step 513: SURGE x4.8 | queue 824 | breaches 0
step 514: SURGE x4.8 | queue 982 | breaches 35
step 515: SURGE x4.8 | queue 1043 | breaches 164
step 516: SURGE x4.8 | queue 1201 | breaches 222
step 517: SURGE x4.8 | queue 1398 | breaches 193
step 518: SURGE x4.8 | queue 1611 | breaches 227

Saved surge_timeseries.json
Peak queue during surge: 4501
Total breaches from surge: 40630
